# E-Commerce Churn & LTV — Phase 3: Data Cleaning + Feature Engineering

**MLDLC stage:** Step 4 (Data Preprocessing) + Step 6 (Feature Engineering), per `MLDLC_best_practices`.

**Built directly from your actual Phase 2 EDA results — not assumptions:**
- 50,000 rows, 26 cols (incl. synthesized `Customer_ID`). Zero exact-duplicate rows.
- Missing values in 14 columns, ranging **0.34% (`Customer_Service_Calls`) to 12.00% (`Social_Media_Engagement_Score`)**. No missing values in any categorical column.
- `Churned`: 35,550 active (71.1%) vs 14,450 churned (28.9%) → **imbalance ratio 2.46 : 1 — mild/moderate, not severe.**
- Strongest churn correlations: `Customer_Service_Calls` (+0.291), `Cart_Abandonment_Rate` (+0.278), `Days_Since_Last_Purchase` (+0.153); `Lifetime_Value` correlates with `Churned` at essentially zero (-0.011) — churners and non-churners have similar historical LTV, which is exactly why the two-model design (churn + LTV, combined by a rule) carries real information instead of being redundant.
- A cluster of engagement columns (`Session_Duration_Avg`, `Pages_Per_Session`, `Mobile_App_Usage`, `Login_Frequency`, `Wishlist_Items`, `Email_Open_Rate`, `Social_Media_Engagement_Score`, `Cart_Abandonment_Rate`) are mutually correlated at 0.6–0.76 — multicollinearity, addressed below with a composite feature.
- `Signup_Quarter` has exactly **4 values**, very evenly split (~12,450–12,560 each) — good for walk-forward validation, but only enough for **3 expanding-window folds** (not a long series).
- `Lifetime_Value`: mean 1440.63, median 1243.42, max 8987.24 — right-skewed (log-transform recommended at training time, not here).

**Rules for this notebook:**
- No modeling. No train/test split (that's Phase 5, after Feast is set up in Phase 4).
- No resampling (SMOTE) here — resampling must be fit only on a training fold, so it belongs inside the Phase 5 training pipeline, never in a shared feature table everyone reads from.
- Output is one canonical `features.parquet` — encoding/scaling stays model-specific and happens in Phase 5's pipelines, not baked in here, so CatBoost keeps its native categorical handling and nothing is scaled twice.


In [1]:
import numpy as np
import pandas as pd
import warnings

warnings.filterwarnings("ignore")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

print("Setup complete.")


Setup complete.


## 1. Load Data

Same source file as Phase 2 — upload it to this Colab session too, then set `DATA_PATH`.

In [2]:
DATA_PATH = "ecommerce_customer_churn_dataset.csv"  # <-- change if needed

df = pd.read_csv(DATA_PATH)
df = df.reset_index().rename(columns={"index": "Customer_ID"})

print("Loaded:", df.shape)


Loaded: (50000, 26)


## 2. Missing Value Imputation

Median imputation for all 14 numeric columns that actually have missing values (confirmed list from your Phase 2 run). Median, not mean, because several of these columns (`Lifetime_Value`-adjacent purchase fields especially) are right-skewed — mean would be pulled by the tail.

In [3]:
columns_with_missing = [
    "Age", "Session_Duration_Avg", "Pages_Per_Session", "Wishlist_Items",
    "Days_Since_Last_Purchase", "Discount_Usage_Rate", "Returns_Rate", "Email_Open_Rate",
    "Customer_Service_Calls", "Product_Reviews_Written", "Social_Media_Engagement_Score",
    "Mobile_App_Usage", "Payment_Method_Diversity", "Credit_Balance",
]

missing_before = df[columns_with_missing].isnull().sum()
print("Missing BEFORE imputation:")
print(missing_before)

medians = {}
for col in columns_with_missing:
    med = df[col].median()
    medians[col] = med
    df[col] = df[col].fillna(med)

missing_after = df[columns_with_missing].isnull().sum()
print("\nMissing AFTER imputation (should all be 0):")
print(missing_after)
assert missing_after.sum() == 0, "Imputation left missing values behind — stop and check."
print("\nAll 14 columns clean. Medians used (for reference / reproducibility):")
print(pd.Series(medians).round(3))


Missing BEFORE imputation:
Age                              2495
Session_Duration_Avg             3399
Pages_Per_Session                3000
Wishlist_Items                   4000
Days_Since_Last_Purchase         3000
Discount_Usage_Rate              3500
Returns_Rate                     4491
Email_Open_Rate                  2528
Customer_Service_Calls            168
Product_Reviews_Written          3500
Social_Media_Engagement_Score    6000
Mobile_App_Usage                 5000
Payment_Method_Diversity         2500
Credit_Balance                   5500
dtype: int64

Missing AFTER imputation (should all be 0):
Age                              0
Session_Duration_Avg             0
Pages_Per_Session                0
Wishlist_Items                   0
Days_Since_Last_Purchase         0
Discount_Usage_Rate              0
Returns_Rate                     0
Email_Open_Rate                  0
Customer_Service_Calls           0
Product_Reviews_Written          0
Social_Media_Engagement_Score    

## 3. Class Imbalance — Decision (not resampling)

Real imbalance ratio is **2.46 : 1**. That's mild/moderate — this is not a 10:1 fraud-style dataset. Literature and practical experience both point the same direction at this ratio: **class-weighting is the primary strategy**, computed from real class counts below. **SMOTE is still trained and compared in Phase 5** (per the plan), but the expectation going in is that class-weighting performs at least as well with less overfitting risk, since it doesn't synthesize any data.

This notebook only **computes and records** the weights — actual resampling/weighting is applied inside the Phase 5 training pipeline, fit on the training fold only.

In [4]:
churn_counts = df["Churned"].value_counts()
n = len(df)
n_classes = 2

# sklearn's 'balanced' formula: n_samples / (n_classes * n_samples_for_class)
class_weight_0 = n / (n_classes * churn_counts[0])
class_weight_1 = n / (n_classes * churn_counts[1])

print("Real class counts:", churn_counts.to_dict())
print(f"Computed balanced class weights -> 0: {class_weight_0:.4f}, 1: {class_weight_1:.4f}")
print("(Phase 5 passes these into class_weight={0: ..., 1: ...} / CatBoost's auto_class_weights='Balanced')")


Real class counts: {0: 35550, 1: 14450}
Computed balanced class weights -> 0: 0.7032, 1: 1.7301
(Phase 5 passes these into class_weight={0: ..., 1: ...} / CatBoost's auto_class_weights='Balanced')


## 4. Outlier Handling — Decision (not row removal)

Phase 2's IQR pass flagged outlier counts up to ~4.9% of rows (`Payment_Method_Diversity`, 2,439 rows) down to ~0.2% (`Discount_Usage_Rate`, 101 rows). None of these are large enough to indicate corrupted data, and `Payment_Method_Diversity`'s high count is a known IQR artifact on a near-discrete, low-cardinality field, not a real anomaly — IQR fences get unreliable on columns with few distinct values.

**Decision: no rows are dropped or capped here.** CatBoost and RandomForest are both reasonably robust to outliers; where it matters (the Logistic Regression baseline), scaling/capping is applied inside that model's own `sklearn.Pipeline` in Phase 5 — not baked into the shared feature table everyone else uses.

## 5. Feature Engineering

Every feature below is justified by a specific number from Phase 2, not a generic RFM template applied blindly.

In [5]:
# Recency / Frequency / Monetary — RFM core, using existing real columns directly
df["Recency"] = df["Days_Since_Last_Purchase"]
df["Frequency"] = df["Total_Purchases"]
df["Monetary_Proxy"] = df["Average_Order_Value"] * df["Total_Purchases"]

# Tenure-normalized activity — catches "long tenure but suddenly quiet"
df["Tenure_Normalized_Activity"] = df["Total_Purchases"] / (df["Membership_Years"] + 1)

print("Correlation of new RFM features with Churned:")
print(df[["Recency", "Frequency", "Monetary_Proxy", "Tenure_Normalized_Activity", "Churned"]].corr()["Churned"])


Correlation of new RFM features with Churned:
Recency                       0.148015
Frequency                    -0.160029
Monetary_Proxy               -0.010833
Tenure_Normalized_Activity   -0.092284
Churned                       1.000000
Name: Churned, dtype: float64


In [6]:
# Engagement composite — addresses the 0.6-0.76 correlated cluster found in Phase 2 EDA.
# Simple mean of z-scored columns: interpretable, no unexplained PCA components for a first pass.
engagement_cluster = [
    "Session_Duration_Avg", "Pages_Per_Session", "Mobile_App_Usage", "Login_Frequency",
    "Wishlist_Items", "Email_Open_Rate", "Social_Media_Engagement_Score",
]

z = (df[engagement_cluster] - df[engagement_cluster].mean()) / df[engagement_cluster].std()
df["Engagement_Composite"] = z.mean(axis=1)

print("Engagement_Composite correlation with Churned:", df[["Engagement_Composite", "Churned"]].corr().iloc[0, 1].round(4))
print("(Original columns are KEPT too — CatBoost/RandomForest handle redundant features fine; ")
print(" this composite mainly helps the Logistic Regression baseline in Phase 5.)")


Engagement_Composite correlation with Churned: -0.254
(Original columns are KEPT too — CatBoost/RandomForest handle redundant features fine; 
 this composite mainly helps the Logistic Regression baseline in Phase 5.)


In [7]:
# Dissatisfaction composite — built from the two STRONGEST churn correlates found in Phase 2:
# Customer_Service_Calls (+0.291) and Cart_Abandonment_Rate (+0.278). Returns_Rate (+0.054) is
# too weak to add real signal here, so it's left out of this composite (still kept as its own column).
dissatisfaction_cols = ["Customer_Service_Calls", "Cart_Abandonment_Rate"]
z2 = (df[dissatisfaction_cols] - df[dissatisfaction_cols].mean()) / df[dissatisfaction_cols].std()
df["Dissatisfaction_Composite"] = z2.mean(axis=1)

print("Dissatisfaction_Composite correlation with Churned:", df[["Dissatisfaction_Composite", "Churned"]].corr().iloc[0, 1].round(4))


Dissatisfaction_Composite correlation with Churned: 0.3435


## 6. LTV Target — Noted, Not Transformed Here

`Lifetime_Value` is right-skewed (mean 1440.63 > median 1243.42, max 8987.24). A `log1p` transform is recommended for the regression target — but that transform happens **inside the Phase 5 training notebook**, right before fitting, with predictions inverted via `expm1`. The raw `Lifetime_Value` column stays untouched here so it's still directly usable for business reporting/dashboards.

## 7. `event_timestamp` for Feast

This dataset is a single snapshot per customer — no repeated observations over time. Feast still requires a timestamp column for its point-in-time join mechanism (Phase 4), so a fixed ingestion timestamp is added now. This is a schema requirement, not a claim of real event-level timing — documented here plainly, same as in `plan.md`.

In [8]:
df["event_timestamp"] = pd.Timestamp.now(tz="UTC").normalize()
print(df["event_timestamp"].iloc[0])


2026-09-08 00:00:00+00:00


## 8. Walk-Forward Fold Plan (documented now, reused as-is in Phase 5)

Exactly 4 signup quarters exist, evenly sized (~12,450–12,560 rows each). That gives **3 expanding-window folds** — not stored as a column, just fixed here so Phase 5 doesn't redefine it differently by accident:

| Fold | Train on | Validate on |
|---|---|---|
| 1 | Q1 | Q2 |
| 2 | Q1 + Q2 | Q3 |
| 3 | Q1 + Q2 + Q3 | Q4 |

Final test set for the one-time end-of-funnel evaluation should be held out from **Q4** (the last quarter), consistent with this same forward-in-time logic — decided here so Phase 5's "test set touched once" rule (Section 9.1 of `plan.md`) has an unambiguous definition.

## 9. Final Assembly & Save

In [9]:
final_columns = [
    "Customer_ID", "event_timestamp",
    # demographics
    "Age", "Gender", "Country", "City", "Membership_Years", "Signup_Quarter",
    # engagement (raw, kept)
    "Login_Frequency", "Session_Duration_Avg", "Pages_Per_Session", "Cart_Abandonment_Rate",
    "Wishlist_Items", "Email_Open_Rate", "Mobile_App_Usage", "Social_Media_Engagement_Score",
    # purchase behavior (raw, kept)
    "Total_Purchases", "Average_Order_Value", "Days_Since_Last_Purchase",
    "Discount_Usage_Rate", "Returns_Rate", "Payment_Method_Diversity",
    # customer service
    "Customer_Service_Calls", "Product_Reviews_Written",
    # financial & status
    "Credit_Balance", "Lifetime_Value", "Churned",
    # engineered
    "Recency", "Frequency", "Monetary_Proxy", "Tenure_Normalized_Activity",
    "Engagement_Composite", "Dissatisfaction_Composite",
]

features_df = df[final_columns].copy()

print("Final shape:", features_df.shape)
print("Total missing values remaining:", int(features_df.isnull().sum().sum()))
assert features_df.isnull().sum().sum() == 0, "Something is still missing — stop and check."

features_df.to_parquet("features.parquet", index=False)
print("Saved: features.parquet")

# In Colab: download it to your machine, then `dvc add data/processed/features.parquet` locally.
try:
    from google.colab import files
    files.download("features.parquet")
except ImportError:
    print("Not running in Colab — file is saved locally in the working directory.")


Final shape: (50000, 33)
Total missing values remaining: 0
Saved: features.parquet


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 10. Summary — copy this block back into chat

In [10]:
summary = {
    "final_shape": features_df.shape,
    "final_columns": list(features_df.columns),
    "missing_values_total": int(features_df.isnull().sum().sum()),
    "class_weights": {"0": round(float(class_weight_0), 4), "1": round(float(class_weight_1), 4)},
    "new_feature_correlations_with_churned": {
        "Recency": round(float(df[["Recency", "Churned"]].corr().iloc[0, 1]), 4),
        "Frequency": round(float(df[["Frequency", "Churned"]].corr().iloc[0, 1]), 4),
        "Monetary_Proxy": round(float(df[["Monetary_Proxy", "Churned"]].corr().iloc[0, 1]), 4),
        "Tenure_Normalized_Activity": round(float(df[["Tenure_Normalized_Activity", "Churned"]].corr().iloc[0, 1]), 4),
        "Engagement_Composite": round(float(df[["Engagement_Composite", "Churned"]].corr().iloc[0, 1]), 4),
        "Dissatisfaction_Composite": round(float(df[["Dissatisfaction_Composite", "Churned"]].corr().iloc[0, 1]), 4),
    },
    "walk_forward_folds": {
        "fold_1": {"train": ["Q1"], "validate": "Q2"},
        "fold_2": {"train": ["Q1", "Q2"], "validate": "Q3"},
        "fold_3": {"train": ["Q1", "Q2", "Q3"], "validate": "Q4"},
        "final_test_set": "Q4",
    },
}

import json
print(json.dumps(summary, indent=2, default=str))


{
  "final_shape": [
    50000,
    33
  ],
  "final_columns": [
    "Customer_ID",
    "event_timestamp",
    "Age",
    "Gender",
    "Country",
    "City",
    "Membership_Years",
    "Signup_Quarter",
    "Login_Frequency",
    "Session_Duration_Avg",
    "Pages_Per_Session",
    "Cart_Abandonment_Rate",
    "Wishlist_Items",
    "Email_Open_Rate",
    "Mobile_App_Usage",
    "Social_Media_Engagement_Score",
    "Total_Purchases",
    "Average_Order_Value",
    "Days_Since_Last_Purchase",
    "Discount_Usage_Rate",
    "Returns_Rate",
    "Payment_Method_Diversity",
    "Customer_Service_Calls",
    "Product_Reviews_Written",
    "Credit_Balance",
    "Lifetime_Value",
    "Churned",
    "Recency",
    "Frequency",
    "Monetary_Proxy",
    "Tenure_Normalized_Activity",
    "Engagement_Composite",
    "Dissatisfaction_Composite"
  ],
  "missing_values_total": 0,
  "class_weights": {
    "0": 0.7032,
    "1": 1.7301
  },
  "new_feature_correlations_with_churned": {
    "Recency": 0.